In [8]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Dropout
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings("ignore")

# ========================= UPDATED CONFIG (more sensitive) =========================
CHANNELS = [f"channel_{i}" for i in range(41, 47)]
WINDOW_SIZE = 40
STEP_SIZE = 5
BATCH_SIZE = 256
EPOCHS = 8
LSTM_UNITS = 64
MAX_TRAIN_SAMPLES = 2_000_000

DATA_DIR = "/Users/alex/Downloads/esa-adb-challenge (1)"
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

tf.random.set_seed(42)
np.random.seed(42)

print("=== LSTM-Autoencoder - More Sensitive Version ===\n")

# Load data
train = pd.read_parquet(f"{DATA_DIR}/train.parquet")
test = pd.read_parquet(f"{DATA_DIR}/test.parquet")

normal_mask = train['is_anomaly'] == 0
normal_data = train[CHANNELS].values[normal_mask]

scaler = StandardScaler()
normal_scaled = scaler.fit_transform(normal_data)
test_scaled = scaler.transform(test[CHANNELS].values)

normal_scaled = normal_scaled[:MAX_TRAIN_SAMPLES]

# Windowing
def make_windows(data, window_size, step):
    n = len(data)
    return np.array([data[i:i+window_size] for i in range(0, n - window_size + 1, step)])

X_normal = make_windows(normal_scaled, WINDOW_SIZE, STEP_SIZE)
X_test = make_windows(test_scaled, WINDOW_SIZE, step=1)

print(f"Normal windows: {X_normal.shape} | Test windows: {X_test.shape}")

# Build Model
inp = Input(shape=(WINDOW_SIZE, len(CHANNELS)))
x = LSTM(LSTM_UNITS, activation='tanh', return_sequences=False)(inp)
x = Dropout(0.3)(x)
x = RepeatVector(WINDOW_SIZE)(x)
x = LSTM(LSTM_UNITS, activation='tanh', return_sequences=True)(x)
x = Dropout(0.3)(x)
out = TimeDistributed(Dense(len(CHANNELS)))(x)

model = Model(inp, out)
model.compile(optimizer='adam', loss='mae')
model.summary()

# Train
model_path = f"{CACHE_DIR}/lstm_ae_sensitive.keras"

if os.path.exists(model_path):
    print("Loading cached model...")
    model = tf.keras.models.load_model(model_path)
else:
    print("Training more sensitive model...")
    model.fit(
        X_normal, X_normal,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        shuffle=True,
        verbose=1
    )
    model.save(model_path)
    print("Model saved!")

# Prediction (lighter)
def get_reconstruction_error(model, windows, batch_size=512):
    pred = model.predict(windows, batch_size=batch_size, verbose=1)
    return np.mean(np.abs(windows - pred), axis=(1, 2))

normal_errors = get_reconstruction_error(model, X_normal)
test_errors = get_reconstruction_error(model, X_test)

scores = np.zeros(len(test_scaled))
for i, err in enumerate(test_errors):
    start = i
    end = min(start + WINDOW_SIZE, len(scores))
    scores[start:end] = np.maximum(scores[start:end], err)

print(f"\nMax score: {scores.max():.5f} | Mean score: {scores.mean():.5f}")

# Try aggressive thresholds
thresholds = [0.20, 0.18, 0.16, 0.14, 0.12, 0.10, 0.08]

for thresh in thresholds:
    predictions = (scores > thresh).astype(int)

    # Post-processing
    from scipy.ndimage import binary_closing
    predictions = binary_closing(predictions, structure=np.ones(5)).astype(int)

    rate = predictions.mean() * 100
    print(f"Threshold {thresh:.3f} → Anomalies: {predictions.sum():,} ({rate:.3f}%)")

    sub = pd.DataFrame({
        'id': test['id'].iloc[WINDOW_SIZE-1:].reset_index(drop=True),
        'anomaly_flag': predictions[WINDOW_SIZE-1:]
    })
    sub.to_parquet(f"{DATA_DIR}/submission_lstm_sens_{thresh:.2f}.parquet", index=False)

print("\nDone. Check the anomaly rates above and choose one to upload to Kaggle.")

=== LSTM-Autoencoder - More Sensitive Version ===

Normal windows: (399993, 40, 6) | Test windows: (521241, 40, 6)


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 40, 6)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ (None, 64)             │        18,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_7 (RepeatVector)  │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ (None, 40, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 40, 6)          │           390 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,590 (201.52 KB)

 Trainable params: 51,590 (201.52 KB)

 Non-trainable params: 0 (0.00 B)

Training more sensitive model...
Epoch 1/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 437s 302ms/step - loss: 0.3095 - val_loss: 0.2509
Epoch 2/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 458s 326ms/step - loss: 0.2757 - val_loss: 0.2444
Epoch 3/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 546s 388ms/step - loss: 0.2714 - val_loss: 0.2391
Epoch 4/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 458s 325ms/step - loss: 0.2691 - val_loss: 0.2391
Epoch 5/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 325s 231ms/step - loss: 0.2675 - val_loss: 0.2398
Epoch 6/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 362s 257ms/step - loss: 0.2662 - val_loss: 0.2391
Epoch 7/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 494s 351ms/step - loss: 0.2652 - val_loss: 0.2373
Epoch 8/8
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 641s 455ms/step - loss: 0.2644 - val_loss: 0.2376
Model saved!
782/782 ━━━━━━━━━━━━━━━━━━━━ 154s 190ms/step
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 179s 176ms/step

Max score: 1.55947 | Mean score: 0.44070
Threshold 0.200 → Anomalies: 521,276 (99.999%)
Threshold 0.180 → Anomalies: 521,276 (99.999%